# [PBT] 화장품 이커머스 ㅣ 06. 범주 묶기 기준 확정

## 이 노트북이 하는 일

`brand`와 `category_id`는 종류가 수백 개다. 그대로 더미변수로 만들면 열이 폭증한다.
**상위 몇 개를 남기고 나머지를 '기타'로 묶을지, 그 기준을 여기서 확정한다.**

> **값을 만들지는 않는다.** 실제 묶기는 ⑤ 모델링용 전처리에서 한다.
> 여기서 기준만 못박는 이유는 **결과를 본 뒤에 유리한 기준을 고르는 일을 막기 위해서**다.

### 계획 보고서에 이미 정해 둔 것

> 각 범주 **최소 1,000건**, 그 미만은 '기타'로 묶는다.

**1,000건을 어느 표에서 세는가.**

| 기준 표 | 행 수 | 1,000건의 비중 |
|---|---|---|
| 원본 로그 | 8,259,159 | 0.012% |
| **분석 대상 테이블** | **108,458** | **0.92%** |

75배 차이가 나므로 결과가 전혀 달라진다.

> 번인 적용으로 대상이 190,698명 → 108,458명이 되면서 1,000건의 상대 비중이
> 0.52% → 0.92%로 올라갔다. 이 변화가 `#06` 판정을 가르는 원인이 된다.

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)             # 폰트 경로가 상대경로라 여기서 실행해야 한다
sys.path.insert(0, BASE)

from pandas import read_csv, to_datetime, DataFrame, set_option

set_option("display.max_columns", 50)

### 2. 경로 지정

In [2]:
OUT  = BASE + r"\output"

### 3. 분석 대상 테이블 불러오기

- 05번에서 목표변수까지 붙인 `02c_y.csv`를 쓴다.
- CSV는 타입을 보존하지 않으므로 01번에서 정한 타입을 되살린다.

> `brand`·`category_id`를 여기서는 **명목형으로 바꾸지 않는다.**
> 범주를 새로 묶으려면 문자열이어야 하고, category 타입은 없는 값을 넣을 수 없기 때문이다.

In [3]:
target = read_csv(OUT + r"\02c_y.csv", encoding="utf-8-sig")

target["t0"]         = to_datetime(target["t0"], format="%Y-%m-%d %H:%M:%S")
target["product_id"] = target["product_id"].astype("int32")

print(f"행 수    : {len(target):,}")
print(f"고객 수  : {target['user_id'].nunique():,}")

행 수    : 108,458
고객 수  : 108,458


## #02. 왜 묶어야 하는가

### 1. 고유값 수 확인

In [4]:
uniq = DataFrame({
    "고유값 수": [target["brand"].nunique(), target["category_id"].nunique()],
}, index=["brand", "category_id"])

uniq

,고유값 수
brand,207
category_id,431


### 2. 그대로 더미로 만들면 몇 열이 되는가

- 더미변수는 기준 범주 하나를 빼고 만든다. 범주가 `n`개면 열은 `n − 1`개다.

In [5]:
b_raw = target["brand"].nunique()
c_raw = target["category_id"].nunique()

print(f"brand       {b_raw:>4}종  →  더미 {b_raw - 1:>4}열")
print(f"category_id {c_raw:>4}종  →  더미 {c_raw - 1:>4}열")
print(f"{'':>16}합계  {b_raw + c_raw - 2:>4}열")

brand        207종  →  더미  206열
category_id  431종  →  더미  430열
                합계   636열


> 108,458행에 더미 700열이 넘어가면 **범주 하나당 표본이 몇백 건밖에 안 된다.**
> 그런 범주의 계수는 우연히 큰 값이 나오기 쉬워 신뢰할 수 없다.

## #03. `brand` 결측을 먼저 '미상'으로 바꾼다

### 1. 결측 확인

> ⚠️ **순서가 중요하다.** 결측 상태로 건수를 세면 그 행들이 집계에서 통째로 빠진다.
> `brand`는 결측이 39%라 **가장 큰 범주가 사라진 채로** 기준을 정하게 된다.

In [6]:
n_na = target["brand"].isna().sum()

print(f"brand 결측 : {n_na:,}건  ({n_na / len(target):.2%})")

brand 결측 : 42,136건  (38.85%)


### 2. '미상'으로 채운다

- 계획 보고서의 방침대로 **삭제하지 않고 하나의 범주로 둔다.**
- 결측률이 39%면 우연히 빠진 것으로 보기 어렵고, 결측 자체가 정보일 수 있다.

In [7]:
brand = target["brand"].astype("string").fillna("미상")
cat   = target["category_id"].astype("string")

print(f"brand 결측 : {brand.isna().sum()}건   (0이어야 정상)")
print(f"brand 범주 : {brand.nunique()}종      ('미상' 포함)")

brand 결측 : 0건   (0이어야 정상)
brand 범주 : 208종      ('미상' 포함)


## #04. 어느 표에서 셀 것인가

**분석 대상 테이블(108,458행) 기준으로 센다.**

| | 원본 로그 기준 | 분석 대상 테이블 기준 |
|---|---|---|
| 1,000건의 의미 | 0.012% — 거의 걸러지지 않는다 | **0.92% — 유효하게 걸러진다** |
| 무엇을 세는가 | 그 브랜드가 **조회·담기·구매된 총 횟수** | 그 브랜드를 **처음 담은 고객 수** |
| 모델에 들어가는 것 | 아니다 | **그렇다** |

계수를 추정하는 것은 **고객 단위 표본**이다. 한 브랜드가 원본에서 5만 번 등장해도
그것이 고객 300명의 반복 행동이라면, 그 범주의 계수는 300명으로 추정된다.
따라서 **모델이 실제로 보는 표본에서 세는 것이 맞다.**

> **→ 결정기록**에 이 판단과 근거를 적는다.

## #05. 기준값별 비교

### 1. 묶기 함수 정의

- 건수가 기준 이상인 범주만 남기고 나머지는 '기타'로 바꾼다.
- **"상위 N개를 고른다"가 아니다.** 기준을 넘는 것을 남기므로 개수는 결과적으로 정해진다.

In [8]:
def group_rare(s, min_count, other="기타"):
    counts = s.value_counts()
    keep = counts[counts >= min_count].index
    return s.where(s.isin(keep), other)

### 2. 기준값별로 어떻게 달라지는가

- 확정값 1,000건 주변을 함께 본다. 그래야 1,000이 특별한 지점인지 알 수 있다.

In [9]:
rows = []

for m in [300, 500, 1000, 2000, 5000]:
    b = group_rare(brand, m)
    c = group_rare(cat, m)

    rows.append({
        "기준(건)": m,
        "brand 범주": b.nunique(),
        "brand 기타%": round((b == "기타").sum() / len(b) * 100, 1),
        "category 범주": c.nunique(),
        "category 기타%": round((c == "기타").sum() / len(c) * 100, 1),
        "더미 열": (b.nunique() - 1) + (c.nunique() - 1),
    })

compare = DataFrame(rows).set_index("기준(건)")
compare

,brand 범주,brand 기타%,category 범주,category 기타%,더미 열
기준(건),,,,,
300,43,9.5,92,21.2,133
500,30,14.6,59,32.7,87
1000,15,23.6,25,54.2,38
2000,12,28.5,11,69.8,21
5000,4,49.9,2,94.4,4


### 3. 설명변수 전체는 몇 열이 되는가

- 명목형 더미에 나머지 변수를 더한 값이다.

| 변수 | 열 수 |
|---|---|
| `cart_timeband` 4구간 | 3 |
| `cart_weekday` 7요일 | 6 |
| `has_prior_view` | 1 |
| 연속형 4개 (`price` · `prior_view_cnt` · `prior_uniq_product_cnt` · `cart_product_view_cnt`) | 4 |
| **소계** | **14** |

In [10]:
OTHERS = 14

compare["X 전체 열"] = compare["더미 열"] + OTHERS
compare[["더미 열", "X 전체 열"]]

,더미 열,X 전체 열
기준(건),,
300,133,147
500,87,101
1000,38,52
2000,21,35
5000,4,18


## #06. 판정

### 1. 판정 기준

| 나온 값 | 어떻게 보는가 |
|---|---|
| '기타' 비중이 **50% 초과** | 통제 효과가 약하다. 기준을 낮출지 검토 |
| X 전체가 **100열 초과** | 108,458행에 부담이다. 기준을 높일지 검토 |
| 둘 다 아니다 | **1,000건 그대로 확정** |

In [11]:
MIN_COUNT = 1000

row = compare.loc[MIN_COUNT]

over_other = max(row["brand 기타%"], row["category 기타%"]) > 50
over_col   = row["X 전체 열"] > 100

print(f"기준 {MIN_COUNT:,}건")
print(f"  brand      {row['brand 범주']:>3}종  기타 {row['brand 기타%']:>5.1f}%")
print(f"  category   {row['category 범주']:>3}종  기타 {row['category 기타%']:>5.1f}%")
print(f"  X 전체     {row['X 전체 열']:>3}열")
print()
print(f"  기타 50% 초과 : {over_other}")
print(f"  100열 초과    : {over_col}")
print()
print("판정 :", "재검토 필요" if (over_other or over_col) else "1,000건 그대로 확정")

기준 1,000건
  brand      15.0종  기타  23.6%
  category   25.0종  기타  54.2%
  X 전체     52.0열

  기타 50% 초과 : True
  100열 초과    : False

판정 : 재검토 필요


> **기준을 바꾸는 경우 조정 전후 값과 사유를 반드시 함께 기록한다.**
> 그것이 "결과를 보고 유리한 기준을 고르는 것"과 갈리는 지점이다.

## #06-2. 재검토

`#06`의 판정이 **재검토 필요**로 나왔다. `category`의 '기타'가 54.2%로 50% 선을 넘었기 때문이다.

원인은 표본 크기 변화다. 번인을 적용하면서 대상이 190,698명 → 108,458명으로 43% 줄었는데
기준값 1,000건은 그대로여서 상대 비중이 0.52% → 0.92%로 두 배 빡빡해졌다.

세 가지를 확인하고 결론을 낸다.

| 확인할 것 | 왜 |
|---|---|
| ① 두 변수의 범주 크기 구조 | 같은 기준을 적용하는 것이 타당한가 |
| ② `category_id`의 계층 구조 | 굵게 묶는 대안이 있는가 |
| ③ `price` 분산을 무엇이 설명하는가 | 품목 통제가 실제로 필요한가 |

> **셋 다 목표변수를 보지 않는다.** 설명변수의 분포와 설명변수끼리의 관계만 본다.
> Y를 본 뒤 기준을 고르면 이 노트북이 첫머리에 못박은 규율을 깨는 것이 된다.

### 1. 두 변수의 범주 크기 구조

In [12]:
rows = []

for name, s in [("brand", brand), ("category_id", cat)]:
    vc = s.value_counts()
    for m in [300, 500, 700, 1000, 1500, 2000]:
        keep = vc[vc >= m]
        rows.append({
            "변수"      : name,
            "기준(건)"  : m,
            "유지 종수" : len(keep),
            "커버율(%)" : round(keep.sum() / len(s) * 100, 1),
            "기타(%)"   : round(100 - keep.sum() / len(s) * 100, 1),
            "최소 범주" : int(keep.min()),
        })

DataFrame(rows).set_index(["변수", "기준(건)"])

유지 종수  커버율(%)  기타(%)  최소 범주
변수          기준(건)                             
brand       300       42    90.5    9.5    315
            500       29    85.4   14.6    512
            700       18    79.4   20.6    735
            1000      14    76.4   23.6   1535
            1500      14    76.4   23.6   1535
            2000      11    71.5   28.5   2067
category_id 300       91    78.8   21.2    308
            500       58    67.3   32.7    500
            700       37    56.0   44.0    702
            1000      24    45.8   54.2   1009
            1500      12    33.4   66.6   1724
            2000      10    30.2   69.8   2161

> **이 표에서 확인할 것**
>
> | 보는 것 | 판단 |
> |---|---|
> | 기준을 올려도 **유지 종수가 그대로인 구간**이 있는가 | 있다면 그 사이에 범주가 하나도 없다는 뜻 — **자연스러운 절단면** |
> | 상위 소수가 커버율을 크게 덮는가 | 소수 대형 구조라면 적은 더미로 잡힌다 |
> | 커버율이 기준에 따라 매끄럽게만 줄어드는가 | 절단면이 없다는 뜻 — 어디를 잘라도 절충이 되지 않는다 |
>
> **채워 넣을 값**
>
> ```
> brand        절단면(유지 종수가 같은 구간) :  1,000건 ~ 1,500건
>              1,000건 기준 →  14종 / 커버 76.4% / 기타 23.6%
>
> category_id  절단면 :  (없음)
>                500건 기준 →  58종 / 커버 67.3% / 기타 32.7%
>              1,000건 기준 →  24종 / 커버 45.8% / 기타 54.2%
>
> 판정 : 두 변수에 같은 기준을 적용할 수 있는가  →  (아니오)
> ```
>
> **구조가 다르면 같은 기준을 적용할 근거가 없다.** 
>
> 따라서 변수별로 따로 정한다.

### 2. `category_id`를 굵게 묶을 수 있는가

익명 ID가 `대분류-중분류-소분류` 식으로 자리를 나눠 쓴다면, 앞자리만 잘라 굵게 묶을 수 있다.
우편번호 앞자리를 자르면 광역시도가 되는 것과 같은 원리다. 그렇다면 더미 몇 개로 통제가 끝난다.

In [13]:
print("자릿수 분포 :", cat.str.len().value_counts().to_dict())
print()

for k in [6, 8, 10, 12, 14]:
    print("앞 %2d자리 기준 고유값 : %4d종" % (k, cat.str[:k].nunique()))

print()
print("값 예시 (정렬 후 앞 4개)")
for v in sorted(cat.unique())[:4]:
    print("   ", v)

자릿수 분포 : {np.int64(19): 108458}

앞  6자리 기준 고유값 :  104종
앞  8자리 기준 고유값 :  108종
앞 10자리 기준 고유값 :  149종
앞 12자리 기준 고유값 :  431종
앞 14자리 기준 고유값 :  431종

값 예시 (정렬 후 앞 4개)
    1487580004832248652
    1487580004857414477
    1487580004882580302
    1487580004916134735


> **이 출력에서 확인할 것**
>
> 계층이라면 어느 자릿수에서 고유값 개수가 **멈춰 있어야** 한다.
> 층이 나뉘어 있으므로 대분류 자리에서 개수가 한동안 평평해야 한다는 뜻이다.
> 자릿수를 늘려도 계단 없이 계속 늘어난다면 계층이 아니다.
>
> **값 예시도 함께 본다.** 앞자리가 같은 값들의 뒷부분이 **일정한 간격으로 증가**한다면,
> 등록 순서대로 번호를 발급한 흔적이지 품목 분류가 아니다.
> 그 경우 앞자리가 같다는 것은 "같은 품목군"이 아니라 **"비슷한 시기에 등록됨"**을 뜻할 뿐이다.
>
> **채워 넣을 값**
>
> ```
> 앞 6 / 8 / 10 / 12자리 고유값 :  ____ / ____ / ____ / ____종
> 계단(층) 존재 여부            :  (있음 / 없음)
> 가장 굵게 잘랐을 때           :  ____종        (목표는 10종 내외)
> 값 예시의 증가 패턴           :  (분류 체계 / 순차 발급)
>
> 판정 : 상위 분류로 묶을 수 있는가  →  (가능 / 불가)
> ```
>
> `category_code`는 결측 98.4%로 이미 제외했다. 여기서도 계층이 없다면
> **품목 계층 정보는 이 데이터에 존재하지 않는다.**
> 그 경우 431종을 통째로 받거나 안 받거나 둘 중 하나다.

### 3. 품목 통제가 실제로 필요한가

빼기로 결론 내기 전에, 통제의 필요성 자체는 확인해 둔다.
`price`의 분산을 무엇이 설명하는지 본다. 목표변수는 쓰지 않는다.

In [14]:
def eta2(g):
    gm   = target["price"].mean()
    ss_t = ((target["price"] - gm) ** 2).sum()
    ss_b = target.groupby(g)["price"].apply(lambda s: len(s) * (s.mean() - gm) ** 2).sum()
    return ss_b / ss_t

DataFrame(
    {"eta^2 (price 분산 설명력)": [round(eta2(cat), 3),
                                  round(eta2(brand), 3),
                                  round(eta2(brand + "|" + cat), 3)]},
    index=["category_id 단독", "brand 단독", "두 변수 조합"],
)

,eta^2 (price 분산 설명력)
category_id 단독,0.652
brand 단독,0.552
두 변수 조합,0.857


> **이 표에서 확인할 것**
>
> `price` 분산을 품목이 크게 설명한다면, 품목 통제 없이 `price` 계수를
> "가격 효과"로 읽어서는 안 된다는 뜻이다. 즉 **통제의 필요성 자체는 실재**한다.
>
> 두 변수의 η²가 각각 높으면서 조합이 그보다 더 높다면, 둘은 대체재가 아니라
> 서로 다른 정보를 담고 있다는 뜻이다.
>
> **채워 넣을 값**
>
> ```
> eta^2   category_id 단독 : _____     brand 단독 : _____     조합 : _____
>
> 판정 : 품목 통제의 필요성  →  (있음 / 없음)
>        brand 가 category 를 대체하는가  →  (예 / 아니오)
> ```

### 4. 재판정

`#06-2 1~3`의 결과를 근거로 결정한다.
`brand`와 `category_id`는 구조와 역할이 다르므로 **따로 결정한다.**

| 변수 | 기준 | 결과 | 모델 투입 |
|---|---|---|---|
| `brand_grp` | **1,000건** | ____종 → 더미 ____열 | **X 포함** |
| `category_grp` | **500건** | ____종 → 더미 ____열 | **X 제외** (대조용) |

**`brand_grp` — 1,000건 확정.**
`#06-2 1`에서 확인한 자연 절단면과 일치하는 지점이다.
실제 브랜드명이라 해석이 되고, '미상'도 "브랜드 표기가 없는 상품"이라는 의미 있는 범주다.
→ 커버율 ____%, '기타' ____%, '미상' ____%

**`category_grp` — X에서 제외.**

제외 사유는 **해석 불가능성이 아니다.** 통제 변수는 계수를 읽지 않으므로
해석 가능성은 판단 기준이 되지 않는다(의료 연구에서 병원 더미 200개를 넣되
"47번 병원의 계수"는 아무도 읽지 않는 것과 같다).
제외하는 이유는 **통제 변수로서 기능하지 못하기 때문**이다.

- 어느 절단 기준에서도 표본의 상당 부분이 '기타'로 뭉친다 (1,000건 ____% / 500건 ____%).
  '기타' 안에서는 통제가 **전혀** 작동하지 않는다. 젤네일과 헤어드라이어가 같은 값을 갖는다.
- `#06-2 2`에서 확인했듯 계층 구조가 없어 굵게 묶는 대안도 없다.
- 결국 더미 ____개를 써서 표본의 ____%만 통제하는 셈이며, 이 비용 대비 효과를 정당화할 근거가 없다.

**`#06`의 "X 100열 초과 시 재검토" 기준은 표본 크기를 모르는 상태에서 세운 것이라 근거가 약하다.**
양성 22,236건 기준 EPV(사건당 변수 수)를 `#07-4`에서 산출한다. 통상 기준은 10이다.
**열 수가 이 표본에서 실제 제약인지, 아니면 통제 효과가 절반뿐인 것이 진짜 제약인지**를 그 값으로 판단하고 기록한다.

### 5. 제외의 타당성은 4일차에 검증한다

품목을 빼면 교란 위험이 남는다.
고관여 상품(예: 헤어 기기)은 오래 보고 덜 사고, 저가 소모품(예: 네일 폴리시)은 덜 보고 더 살 수 있다.
그렇다면 **"조회를 많이 하면 덜 산다"는 가짜 결론**이 나오는데, 그것이 이 프로젝트의 핵심 질문이다.

그래서 `category_grp` 컬럼을 08번 마스터 테이블에 **함께 담아두고**(더미는 만들지 않는다),
4일차 기준선 모형에서 포함 버전을 한 번 더 적합해 대조한다.

**판정 기준 — 지금 확정하며 이후 변경하지 않는다.**

> 같은 층화 분할·같은 전처리로 두 모형을 적합한다. **X 목록 한 줄만 다르다.**
>
> - 모형 A = `category_grp` 제외
> - 모형 B = `category_grp` 포함
>
> **핵심 X 5개** — `has_prior_view` · `prior_view_cnt` · `prior_uniq_product_cnt` ·
> `cart_product_view_cnt` · `price` — **의 계수 부호와 유의성 판정(α = 0.05)이
> 두 모형에서 모두 같으면 제외를 확정한다.** 하나라도 다르면 포함 버전을 채택하고 사유를 기록한다.
>
> **AUC 등 성능 지표는 이 판정에 사용하지 않는다.** 성능이 좋은 쪽을 고르는 순간 규율 위반이다.
>
> 부호·유의성은 같으나 계수 크기가 30% 이상 변한 변수가 있으면, 판정은 통과로 하되 그 사실을 기록한다.

대조용을 500건(더 촘촘한 통제)으로 잡는 이유 — **가장 강하게 통제한 버전에서도 결론이 바뀌지 않는다면
그보다 약한 어떤 버전에서도 바뀌지 않는다.** 약한 버전으로 대조하면
"더 세게 넣었으면 달랐을 수도 있다"는 반박이 남는다.

> **교재 근거** — LAB-13은 공간자기상관에서 이웃 정의를 Queen·Rook 두 가지로 계산해
> "이웃 정의를 바꿔도 I = 0.790, p = 0.001로 결론 동일"을 확인했다.
> 설계 선택 하나를 바꿔 결론의 견고함을 보는 같은 절차다.

### 6. 보고서에 쓸 문장 (수치는 위 결과로 채운다)

> 익명 `category_id` ____종은 계층 구조가 없어 상위 분류로 묶을 수 없었고,
> 범주 크기의 중앙값이 ____명이라 어떤 절단 기준에서도 표본의 ____% 이상이 '기타'로 뭉쳐
> 통제 변수로 기능하지 못했다. 품목 통제는 `brand_grp`(____개 범주, 커버율 ____%)로 대신했으며,
> `category_grp` 포함 모형과의 민감도 비교에서 핵심 설명변수의 계수 부호와 유의성이 동일함을 확인했다.

> ⚠️ **"익명이라 해석이 안 돼서 뺐다"고 쓰지 않는다.** 통제 변수는 해석하는 변수가 아니므로
> 그 논거는 "통제 변수는 해석하는 게 아닙니다" 한 마디에 무너진다. 위 문장은 데이터의 성질을 근거로 삼는다.

> **`price` 해석의 수위** — 품목 통제가 불완전하므로 "가격이 전환을 낮춘다"가 아니라
> **"가격대가 높은 집단에서 전환율이 낮게 관측되었다"**로 진술한다.

## #07. 묶은 결과 확인

### 1. `brand` — 큰 범주부터

In [15]:
BRAND_MIN = 1000    # 데이터의 자연 절단면        (#06-2 1)
CAT_MIN   =  500    # 민감도 대조용, 최대 통제    (#06-2 5)

b_grp = group_rare(brand, BRAND_MIN)

b_tbl = b_grp.value_counts().rename("고객 수").to_frame()
b_tbl["비율 %"] = (b_tbl["고객 수"] / len(b_grp) * 100).round(2)
b_tbl.head(12)

,고객 수,비율 %
brand,,
미상,42136,38.85
기타,25627,23.63
runail,6744,6.22
irisk,5457,5.03
grattol,4741,4.37
estel,3144,2.9
masura,3062,2.82
kapous,2764,2.55
italwax,2713,2.5


### 2. '미상'과 '기타'가 어디에 있는가

- 이 둘은 실제 브랜드가 아니라 **처리 결과로 생긴 범주**다. 크기를 알고 있어야 해석할 때 헷갈리지 않는다.

In [16]:
for name in ["미상", "기타"]:
    n = (b_grp == name).sum()
    rank = list(b_tbl.index).index(name) + 1 if name in b_tbl.index else None
    print(f"{name} : {n:>7,}명  ({n / len(b_grp):>6.2%})   순위 {rank}")

미상 :  42,136명  (38.85%)   순위 1
기타 :  25,627명  (23.63%)   순위 2


### 3. `category_id` — 큰 범주부터

- 익명 숫자라 개별 해석은 하지 않는다. 통제 목적으로만 만든다.
- **기준은 500건이다**(`CAT_MIN`). 대조용이므로 가장 촘촘한 통제로 잡는다 — `#06-2 5` 참조.
- 이 변수는 **모델 X에 넣지 않는다.** 4일차 민감도 검증에서만 꺼내 쓴다.

In [17]:
c_grp = group_rare(cat, CAT_MIN)

c_tbl = c_grp.value_counts().rename("고객 수").to_frame()
c_tbl["비율 %"] = (c_tbl["고객 수"] / len(c_grp) * 100).round(2)
c_tbl.head(10)

,고객 수,비율 %
category_id,,
기타,35473,32.71
1487580005092295511,6102,5.63
1487580008246412266,4260,3.93
1602943681873052386,3514,3.24
1487580006317032337,3494,3.22
1487580013841613016,3109,2.87
1487580007675986893,2856,2.63
1487580005268456287,2656,2.45
1487580005595612013,2339,2.16


### 4. 남는 범주 수 최종 확인

In [18]:
print(f"brand_grp    : {b_grp.nunique():>3}종  →  더미 {b_grp.nunique() - 1:>3}열   [X 포함]")
print(f"category_grp : {c_grp.nunique():>3}종  →  더미 {c_grp.nunique() - 1:>3}열   [X 제외 · 대조용]")
print()
print(f"모형 A (기본)   X 전체 : {b_grp.nunique() - 1 + OTHERS:>3}열")
print(f"모형 B (대조용) X 전체 : {b_grp.nunique() + c_grp.nunique() - 2 + OTHERS:>3}열")
print()
print(f"양성 22,236건 기준 EPV : A {22236 / (b_grp.nunique() - 1 + OTHERS):.0f}"
      f"  ·  B {22236 / (b_grp.nunique() + c_grp.nunique() - 2 + OTHERS):.0f}"
      f"   (통상 기준 10)")

brand_grp    :  15종  →  더미  14열   [X 포함]
category_grp :  59종  →  더미  58열   [X 제외 · 대조용]

모형 A (기본)   X 전체 :  28열
모형 B (대조용) X 전체 :  86열

양성 22,236건 기준 EPV : A 794  ·  B 259   (통상 기준 10)
